# 06. Transcript Cleaning

## Goal

Turn noisy timestamped ASR rows into cleaner transcript rows that are easier to read, search, and summarize.

## Why This Step Matters

Long-audio ASR output can contain extra spaces, empty chunks, and repeated tokens. Cleaning fixes those simple issues while keeping timestamps and meaning intact.

## Inputs

- Timestamped transcript rows from the long-meeting transcription workflow.
- Cleaning utilities from `src/transcript/clean.py`.

## Outputs

- Cleaned transcript rows.
- Optional local JSON and Markdown files under ignored `outputs/`.

## Setup

Keep this notebook small: use sample rows first, then point it at saved ASR rows later if needed.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.transcript.clean import clean_transcript_rows

SAVE_OUTPUTS = False
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "transcripts"

print("Project root:", PROJECT_ROOT)

Project root: /home/lataeq/Code/Saranote


## Load Data

Start with a tiny transcript sample that has the same shape as Phase 6 output.

In [2]:
raw_rows = [
    {"timestamp": "00:00:00 - 00:00:06.830", "text": "  สวัสดีครับ   สวัสดีครับ   "},
    {"timestamp": "00:00:06.830 - 00:00:12.000", "text": "ครับ ครับ ครับ ที่ประชุมเห็นชอบ"},
    {"timestamp": "00:00:12.000 - 00:00:15.000", "text": "   "},
]

pd.DataFrame(raw_rows)

,timestamp,text
0,00:00:00 - 00:00:06.830,สวัสดีครับ สวัสดีครับ
1,00:00:06.830 - 00:00:12.000,ครับ ครับ ครับ ที่ประชุมเห็นชอบ
2,00:00:12.000 - 00:00:15.000,


## Run Experiment

Clean whitespace, drop empty rows, and trim long runs of identical adjacent tokens.

In [3]:
cleaned_rows = clean_transcript_rows(raw_rows)
cleaned_df = pd.DataFrame(cleaned_rows)
cleaned_df

,timestamp,text
0,00:00:00 - 00:00:06.830,สวัสดีครับ สวัสดีครับ
1,00:00:06.830 - 00:00:12.000,ครับ ครับ ที่ประชุมเห็นชอบ


## Inspect Results

Compare raw and cleaned text side by side before using the cleaned rows downstream.

In [4]:
comparison = pd.DataFrame({
    "timestamp": [row["timestamp"] for row in raw_rows],
    "raw_text": [row["text"] for row in raw_rows],
    "cleaned_text": [
        next((cleaned["text"] for cleaned in cleaned_rows if cleaned["timestamp"] == row["timestamp"]), "")
        for row in raw_rows
    ],
})
comparison

,timestamp,raw_text,cleaned_text
0,00:00:00 - 00:00:06.830,สวัสดีครับ สวัสดีครับ,สวัสดีครับ สวัสดีครับ
1,00:00:06.830 - 00:00:12.000,ครับ ครับ ครับ ที่ประชุมเห็นชอบ,ครับ ครับ ที่ประชุมเห็นชอบ
2,00:00:12.000 - 00:00:15.000,,


## Cleaning Checks

Run a few small checks so it is clear what the cleaner is doing well and what it intentionally leaves alone.

In [5]:
test_cases = [
    {
        "case": "collapse whitespace",
        "raw_text": "  สวัสดี   ครับ\nทุกคน  ",
        "expected_text": "สวัสดี ครับ ทุกคน",
    },
    {
        "case": "trim long adjacent repeats",
        "raw_text": "ครับ ครับ ครับ ครับ",
        "expected_text": "ครับ ครับ",
    },
    {
        "case": "keep one natural repeat",
        "raw_text": "สวัสดีครับ สวัสดีครับ",
        "expected_text": "สวัสดีครับ สวัสดีครับ",
    },
    {
        "case": "drop empty segment",
        "raw_text": "   ",
        "expected_text": "",
    },
    {
        "case": "keep non-adjacent repeat",
        "raw_text": "มติ เห็นชอบ มติ",
        "expected_text": "มติ เห็นชอบ มติ",
    },
]

test_rows = [
    {"timestamp": f"test-{index:02d}", "text": case["raw_text"]}
    for index, case in enumerate(test_cases, start=1)
]
cleaned_test_rows = clean_transcript_rows(test_rows)
cleaned_by_timestamp = {row["timestamp"]: row["text"] for row in cleaned_test_rows}

check_rows = []
for index, case in enumerate(test_cases, start=1):
    timestamp = f"test-{index:02d}"
    actual_text = cleaned_by_timestamp.get(timestamp, "")
    check_rows.append({
        "case": case["case"],
        "raw_text": case["raw_text"],
        "expected_text": case["expected_text"],
        "actual_text": actual_text,
        "passed": actual_text == case["expected_text"],
    })

checks_df = pd.DataFrame(check_rows)
print(f"Passed {checks_df['passed'].sum()}/{len(checks_df)} cleaning checks")
checks_df

Passed 5/5 cleaning checks


,case,raw_text,expected_text,actual_text,passed
0,collapse whitespace,สวัสดี ครับ\nทุกคน,สวัสดี ครับ ทุกคน,สวัสดี ครับ ทุกคน,True
1,trim long adjacent repeats,ครับ ครับ ครับ ครับ,ครับ ครับ,ครับ ครับ,True
2,keep one natural repeat,สวัสดีครับ สวัสดีครับ,สวัสดีครับ สวัสดีครับ,สวัสดีครับ สวัสดีครับ,True
3,drop empty segment,,,,True
4,keep non-adjacent repeat,มติ เห็นชอบ มติ,มติ เห็นชอบ มติ,มติ เห็นชอบ มติ,True


In [6]:
assert checks_df["passed"].all(), "Some transcript cleaning checks failed"

cleanup_summary = pd.DataFrame([
    {
        "raw_rows": len(raw_rows),
        "cleaned_rows": len(cleaned_rows),
        "empty_rows_removed": len(raw_rows) - len(cleaned_rows),
        "raw_characters": sum(len(row["text"]) for row in raw_rows),
        "cleaned_characters": sum(len(row["text"]) for row in cleaned_rows),
    }
])

cleanup_summary

,raw_rows,cleaned_rows,empty_rows_removed,raw_characters,cleaned_characters
0,3,2,1,62,47


## Optional Save

`outputs/` is ignored, so saved transcript artifacts stay local by default.

In [7]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "cleaned_transcript.json").write_text(
        json.dumps(cleaned_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    cleaned_markdown = "\n".join(
        f"- [{row['timestamp']}] {row['text']}" for row in cleaned_rows
    )
    (OUTPUT_DIR / "cleaned_transcript.md").write_text(cleaned_markdown, encoding="utf-8")
    print("Saved cleaned transcript files to:", OUTPUT_DIR)